# NB_lab_test_trends

Builds the **gold.lab_test_trends** table — one row per test type per month with abnormal and critical rates plus month-over-month abnormal-rate trend via `LAG()`. Full rebuild every run.

In [0]:
# -- Configuration ----------------------------------------------------------
catalog = "adb_databeli"
target_table = f"{catalog}.gold.lab_test_trends"

print(f"✅ Configuration")
print(f"   📌 Catalog      : {catalog}")
print(f"   📌 Target table : {target_table}")
print(f"   📌 Load type    : FULL (CREATE OR REPLACE)")

In [0]:
%sql
CREATE OR REPLACE TABLE adb_databeli.gold.lab_test_trends AS

WITH monthly_base AS (
    SELECT
        lr.test_name,
        YEAR(lr.test_date)                                                 AS year,
        MONTH(lr.test_date)                                                AS month,
        c.month_name,
        c.quarter_name,
        COUNT(*)                                                           AS total_tests,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)          AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)          AS critical_test_count
    FROM adb_databeli.silver.lab_results  lr
    LEFT JOIN adb_databeli.silver.calendar  c
        ON c.date = DATE_TRUNC('month', lr.test_date)
    GROUP BY
        lr.test_name,
        YEAR(lr.test_date),
        MONTH(lr.test_date),
        c.month_name,
        c.quarter_name
), monthly_rates AS (
    SELECT
        test_name,
        year,
        month,
        month_name,
        quarter_name,
        total_tests,
        abnormal_test_count,
        critical_test_count,
        CAST(ROUND(abnormal_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
        CAST(ROUND(critical_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct
    FROM monthly_base
)

SELECT
    test_name,
    year,
    month,
    month_name,
    quarter_name,
    total_tests,
    abnormal_test_count,
    critical_test_count,
    abnormal_rate_pct,
    critical_rate_pct,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month)    AS prev_month_abnormal_rate_pct,
    CAST(
        ROUND(
            abnormal_rate_pct - LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month),
            2
        ) AS DECIMAL(6,2)
    )                                                                             AS abnormal_rate_change_pct
FROM monthly_rates

In [0]:
row_count = spark.table(target_table).count()
print(f"✅ Table '{target_table}' built successfully")
print(f"   📊 Total rows: {row_count:,}")

display(spark.table(target_table).limit(10))

In [0]:
dbutils.notebook.exit(str(row_count))